# ETL Project with Google Sheets Data

## Accessing Google APIs

There are two ways to authenticate a Service Account in Python:
- oauth2client (OLD)
- google-auth (MODERN)

Both work with gspread, but they differ in design, maintenance, and strictness.

| feature | oauth2client | google-auth |
|--- |--- |--- |
| Status | Deprecated since around 2019; no new features. | Actively maintained and the standard. |
| Ownership | Muddled ownership led to maintenance issues | Owned by the teams maintaining Google Cloud Client Libraries, gRPC, and samples |

## Reading Data from Google Sheet using `oauth2client` (OLD)

In [1]:
# Import Required Libraries
import gspread
from oauth2client.service_account import ServiceAccountCredentials

from dotenv import load_dotenv
import os

load_dotenv() # ← IMPORTANT

# Define scope
scope = ["https://www.googleapis.com/auth/spreadsheets.readonly"]

# Load credentials from JSON
creds = ServiceAccountCredentials.from_json_keyfile_name(
    os.getenv("GOOGLE_CREDS"),
    scope
)

# Authorize client
client = gspread.authorize(creds)

# Open the sheet
sheet = client.open_by_key(os.getenv("GOOGLE_SPREAD_SHEET_ID")).worksheet("Population")

# Read data from the sheet
# head = 2 tells gspread that header row is Row 2
data = sheet.get_all_records(head=2)

#print(data)

## Reading Data from Google Sheet using `google-auth` (MODERN)

In [2]:
import gspread
from google.oauth2.service_account import Credentials
from dotenv import load_dotenv
import os

load_dotenv()

SCOPES = [
    "https://www.googleapis.com/auth/spreadsheets",
    "https://www.googleapis.com/auth/drive"
]

creds = Credentials.from_service_account_file(
    os.getenv("GOOGLE_CREDS"),
    scopes=SCOPES
)

client = gspread.authorize(creds)

sheet = client.open_by_key(os.getenv("GOOGLE_SPREAD_SHEET_ID"))

ws = sheet.worksheet("Population")
raw_data = ws.get("A3:AC197")
headers = ws.row_values(2)

data = [dict(zip(headers, row)) for row in raw_data]

# print(data)

# for row in data:
#    print(row)

In [3]:
for i, row in enumerate(data[:10]):
    print(i, row["Loco Number"])

0 30631
1 30622
2 30630
3 37015
4 39140
5 37370
6 37368
7 32880
8 32787
9 32882


In [4]:
len(data)

195

## Clean Headers

In [5]:
import re

def clean_header(name):
    # lowercase
    name = name.lower()
    
    # replace spaces and special chars with underscore
    name = re.sub(r'[^a-z0-9]+', '_', name)
    
    
    # remove leading/trailing underscores
    name = name.strip('_')
    
    return name

`re.sub(pattern, replacement, string)` replaces parts of a `string` that match a regex `pattern` with `replacement`.

- `r'...'` → raw string (so \ is treated literally)
- `[...]` → character class (match any ONE character inside it)
- `^` inside `[]` → NOT (negation)
- `a-z` → all lowercase letters
- `0-9` → digits
- `[^a-z0-9]` → match anything that is NOT a lowercase letter or digit
- `+` → match one or more of those characters

Match one or more characters that are NOT lowercase letters or digits.

In [6]:
def clean_value(v):
    if isinstance(v, str):
        v = v.strip()
        if v.upper() == "YES":
            return 1
        elif v.upper() == "NO":
            return 0
        elif v in ["", "N/A"]:
            return None
    return v

In [7]:
cleaned_data = [
    {
        clean_header(k): clean_value(v)
        for k, v in row.items()
    }
    for row in data
]

In [8]:
len(cleaned_data)

195

## Save to Database

In [9]:
import mysql.connector
from datetime import datetime

from dotenv import load_dotenv
import os

In [10]:
# -----------------------------
# 1. DB Connection
# -----------------------------
conn = mysql.connector.connect(
    host=os.getenv("MYSQL_HOST"),
    user=os.getenv("MYSQL_USER"),
    password=os.getenv("MYSQL_PASSWORD")
)

cursor = conn.cursor()

In [11]:
# -----------------------------
# 2. Create & Select Database
# -----------------------------
cursor.execute("CREATE DATABASE IF NOT EXISTS kjm_elect_locos_db")
cursor.execute("USE kjm_elect_locos_db")

In [12]:
# -----------------------------
# 3. Create Table
# -----------------------------
create_table_query = """
CREATE TABLE IF NOT EXISTS locomotives (
    sl_no INT,
    loco_number INT PRIMARY KEY,
    type VARCHAR(50),
    date_of_commissioning DATE,
    production_unit VARCHAR(50),
    transformer VARCHAR(50),
    traction_converter VARCHAR(50),
    auxilliary_converter VARCHAR(50),
    vcb VARCHAR(50),
    hog_availability BOOLEAN,
    hog_make VARCHAR(50),
    cab_ac_availability BOOLEAN,
    cab_ac_make VARCHAR(50),
    brake_system VARCHAR(50),
    rtis_dpwcs_availability BOOLEAN,
    type_of_system VARCHAR(50),
    rtis_make VARCHAR(50),
    led_head_lights_available BOOLEAN,
    h_l_make VARCHAR(50),
    led_signal_exchange_lights_availability BOOLEAN,
    make_of_signal_exch_light VARCHAR(50),
    hlc_ivc_type VARCHAR(50)
)
"""

cursor.execute(create_table_query)

In [13]:
# -----------------------------
# 3. Insert Query
# -----------------------------
insert_query = """
INSERT INTO locomotives (
    sl_no, loco_number, type, date_of_commissioning,
    production_unit, transformer, traction_converter, auxilliary_converter,
    vcb, hog_availability, hog_make,
    cab_ac_availability, cab_ac_make,
    brake_system, rtis_dpwcs_availability,
    type_of_system, rtis_make,
    led_head_lights_available, h_l_make,
    led_signal_exchange_lights_availability,
    make_of_signal_exch_light, hlc_ivc_type
)
VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
"""

In [14]:
from datetime import datetime

def parse_date(value):
    if not value:
        return None
    
    value = str(value).strip()
    
    try:
        return datetime.strptime(value, "%d-%m-%Y").date()
    except ValueError:
        return None   # handle "Under Comm", "N/A", etc.


def prepare_row(row):
    return (
        row.get('sl_no'),
        row.get('loco_number'),
        row.get('type'),
        parse_date(row.get('date_of_commissioning')),
        row.get('production_unit'),
        row.get('transformer'),
        row.get('traction_converter'),
        row.get('auxilliary_converter'),
        row.get('vcb'),
        row.get('hog_availability'),
        row.get('hog_make'),
        row.get('cab_ac_availability'),
        row.get('cab_ac_make'),
        row.get('brake_system'),
        row.get('rtis_dpwcs_availability'),
        row.get('type_of_system'),
        row.get('rtis_make'),
        row.get('led_head_lights_available'),
        row.get('h_l_make'),
        row.get('led_signal_exchange_lights_availability'),
        row.get('make_of_signal_exch_light'),
        row.get('hlc_ivc_type')
    )

In [15]:
# -----------------------------
# 5. Bulk Insert (IMPORTANT)
# -----------------------------
rows_to_insert = [prepare_row(row) for row in cleaned_data]

cursor.executemany(insert_query, rows_to_insert)

In [16]:
# -----------------------------
# 6. Commit
# -----------------------------
conn.commit()

print(f"✅ Inserted {cursor.rowcount} rows successfully!")

✅ Inserted 195 rows successfully!


In [17]:
# -----------------------------
# 7. Close
# -----------------------------
cursor.close()
conn.close()